In [1]:
# Parameters
run_id = "130b4b0c-059c-4a33-880b-6e04c41ce18c"
artifacts_dir = "/home/adnoman/projects/aml_gan/AMLend2end/artifacts/runs/130b4b0c-059c-4a33-880b-6e04c41ce18c"
sample_size = None
epochs = None
threshold = None


### Create training dataset for anomaly detection model
In this notebook We are going to create training dataset from node embeddings feature group and register to Hopsworks Feature Store. 
![Training Dataset](./images/create_training_dataset.png)

### Create a connection to hsfs

In [2]:
# Setup for local execution
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Define paths
BASE_PATH = os.path.dirname(os.path.abspath("__file__"))
OUTPUT_PATH = os.path.join(BASE_PATH, "output")
TRAINING_DATA_PATH = os.path.join(BASE_PATH, "training_data")

print(f"Output path: {OUTPUT_PATH}")
print(f"Training data path: {TRAINING_DATA_PATH}")

Output path: /home/adnoman/projects/aml_gan/AMLend2end/output
Training data path: /home/adnoman/projects/aml_gan/AMLend2end/training_data


### Retrieve alert nodes feature group from hsfs

In [3]:
# Load node embeddings feature group (created in notebook 5)
node_embeddings_fg = pd.read_parquet(os.path.join(OUTPUT_PATH, "node_embeddings_fg.parquet"))

print(f"Loaded node embeddings: {node_embeddings_fg.shape}")
print(f"Columns: {node_embeddings_fg.columns.tolist()[:5]}... + is_sar")
node_embeddings_fg.head()

Loaded node embeddings: (7347, 34)
Columns: ['id', 'emb_0', 'emb_1', 'emb_2', 'emb_3']... + is_sar


,id,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31,is_sar
0,3aa9646b,0.006354,-0.020029,0.000121,0.001682,0.022640,0.022485,-0.022633,-0.018184,-0.013343,...,0.013926,0.027505,-0.010198,0.025302,-0.008416,0.009119,0.013350,0.022079,-0.029613,0
1,1e46e726,0.002481,-0.007734,0.013989,-0.021754,0.023876,0.024569,0.005226,-0.014404,-0.014850,...,-0.014103,0.029731,-0.024120,0.001362,0.002191,-0.000155,-0.023380,0.027062,0.014415,0
2,49203bc3,0.024728,-0.005727,0.011389,-0.003658,0.005619,0.027766,0.028874,-0.025506,-0.024554,...,-0.002589,0.025417,0.011257,-0.031272,0.029479,0.019596,-0.007734,0.025463,0.013300,0
3,a74d1101,0.006074,-0.028726,-0.022943,0.030213,0.001226,-0.016350,-0.007838,-0.003526,-0.005832,...,-0.011728,-0.017198,-0.015983,0.005139,0.021491,-0.009442,0.014519,-0.013948,-0.029704,1
4,616d4505,-0.020141,-0.024803,-0.019287,-0.006866,-0.015812,0.008584,-0.000066,0.009143,-0.012181,...,-0.023667,0.026482,-0.003340,-0.031200,-0.025585,0.000395,-0.019562,-0.010542,0.011896,0


### Prepare training datasets for anomaly detection 
###### In the next notebook we are going to train [gan for anomaly detection](https://arxiv.org/pdf/1905.11034.pdf). Durring training step  we will provide only features of accounts that have never been reported for money laundering behaviour.  But we will disclose previously reported accounts to the model only in evaluation step.   

In [4]:
# Get embedding columns
emb_cols = [c for c in node_embeddings_fg.columns if c.startswith('emb_')]
print(f"Embedding dimensions: {len(emb_cols)}")

# Filter non-SAR nodes for training (is_sar == 0)
non_sar_df = node_embeddings_fg[node_embeddings_fg['is_sar'] == 0].copy()
print(f"Non-SAR nodes: {len(non_sar_df)}")

Embedding dimensions: 32
Non-SAR nodes: 6531


In [5]:
# Preview non-SAR embeddings
non_sar_df[emb_cols[:5] + ['is_sar']].head()

,emb_0,emb_1,emb_2,emb_3,emb_4,is_sar
0,0.006354,-0.020029,0.000121,0.001682,0.022640,0
1,0.002481,-0.007734,0.013989,-0.021754,0.023876,0
2,0.024728,-0.005727,0.011389,-0.003658,0.005619,0
4,-0.020141,-0.024803,-0.019287,-0.006866,-0.015812,0
6,0.002133,0.028336,-0.029831,-0.030739,-0.030580,0


In [6]:
print(f"Non-SAR count: {len(non_sar_df)}")

Non-SAR count: 6531


In [7]:
# Create train/test split for non-SAR data (80/20 split)
non_sar_train, non_sar_test = train_test_split(
    non_sar_df, 
    test_size=0.2, 
    random_state=42
)

print(f"Non-SAR train: {len(non_sar_train)}")
print(f"Non-SAR test: {len(non_sar_test)}")

# Extract embeddings as numpy arrays
X_train = non_sar_train[emb_cols].values
y_train = non_sar_train['is_sar'].values

X_test_non_sar = non_sar_test[emb_cols].values
y_test_non_sar = non_sar_test['is_sar'].values

print(f"\nTraining data shape: {X_train.shape}")
print(f"Test data shape: {X_test_non_sar.shape}")

Non-SAR train: 5224
Non-SAR test: 1307

Training data shape: (5224, 32)
Test data shape: (1307, 32)


## For testing and evaluation we will include known SAR nodes to measure anomaly score  

In [8]:
# For evaluation, we need both SAR and non-SAR test data
# Get SAR nodes
sar_df = node_embeddings_fg[node_embeddings_fg['is_sar'] == 1].copy()
print(f"SAR nodes: {len(sar_df)}")

SAR nodes: 816


In [9]:
# Extract SAR embeddings
X_sar = sar_df[emb_cols].values
y_sar = sar_df['is_sar'].values

print(f"SAR data shape: {X_sar.shape}")

SAR data shape: (816, 32)


In [10]:
# Create evaluation dataset: combine non-SAR test + all SAR nodes
X_eval = np.vstack([X_test_non_sar, X_sar])
y_eval = np.concatenate([y_test_non_sar, y_sar])

print(f"Evaluation dataset shape: {X_eval.shape}")
print(f"Evaluation labels: {len(y_eval)} (SAR: {y_eval.sum()}, Non-SAR: {(y_eval==0).sum()})")

Evaluation dataset shape: (2123, 32)
Evaluation labels: 2123 (SAR: 816, Non-SAR: 1307)


In [11]:
print(f"Non-SAR test count: {len(X_test_non_sar)}")

Non-SAR test count: 1307


In [12]:
print(f"SAR count: {len(X_sar)}")

SAR count: 816


In [13]:
print(f"Evaluation total count: {len(X_eval)}")

Evaluation total count: 2123


In [14]:
# Save training datasets locally (replaces hsfs tfrecord)
GAN_DATA_PATH = os.path.join(TRAINING_DATA_PATH, "gan")
os.makedirs(GAN_DATA_PATH, exist_ok=True)

# Save as numpy arrays (efficient for training)
np.save(os.path.join(GAN_DATA_PATH, "X_train.npy"), X_train)
np.save(os.path.join(GAN_DATA_PATH, "y_train.npy"), y_train)
np.save(os.path.join(GAN_DATA_PATH, "X_eval.npy"), X_eval)
np.save(os.path.join(GAN_DATA_PATH, "y_eval.npy"), y_eval)

print(f"Saved training data to: {GAN_DATA_PATH}")
print(f"  - X_train.npy: {X_train.shape}")
print(f"  - y_train.npy: {y_train.shape}")
print(f"  - X_eval.npy: {X_eval.shape}")
print(f"  - y_eval.npy: {y_eval.shape}")

Saved training data to: /home/adnoman/projects/aml_gan/AMLend2end/training_data/gan
  - X_train.npy: (5224, 32)
  - y_train.npy: (5224,)
  - X_eval.npy: (2123, 32)
  - y_eval.npy: (2123,)


## Training dataset provenance
![Training dataset provenance](./images/provenance_td.png)

In [15]:
# Summary
print("=" * 50)
print("GAN Training Datasets Created")
print("=" * 50)
print(f"Training set (non-SAR only): {X_train.shape[0]} samples")
print(f"Evaluation set (SAR + non-SAR): {X_eval.shape[0]} samples")
print(f"  - Non-SAR: {(y_eval==0).sum()}")
print(f"  - SAR: {y_eval.sum()}")
print(f"Embedding dimensions: {X_train.shape[1]}")
print(f"\nSaved to: {GAN_DATA_PATH}")
print("=" * 50)

GAN Training Datasets Created
Training set (non-SAR only): 5224 samples
Evaluation set (SAR + non-SAR): 2123 samples
  - Non-SAR: 1307
  - SAR: 816
Embedding dimensions: 32

Saved to: /home/adnoman/projects/aml_gan/AMLend2end/training_data/gan
